# 02 · Silver layer: cleaned and standardized data



**Purpose.** Clean and standardize the Bronze data with Spark DataFrame transformations.

**Inputs.** `workspace.car_insurance.claims_bronze` (name in `BRONZE_TABLE`).

**Outputs.** `workspace.car_insurance.claims_silver` (name in `SILVER_TABLE`).

**Assumptions.** 
- A column named `_corrupt_record` was created in the Bronze Layer

**What the next notebook relies on (the table contract)**

- The 19 source columns, same names, one row per driver. `ID` must stay unique and non-null: it joins drivers to their Phase 2 split.
- Numeric columns stored as numbers and category columns as text. Keep `POSTAL_CODE` as text.
- **Do not impute, scale or encode here.** Phase 2 fitted those steps on training drivers only; doing them on all rows leaks test data. They happen inside each model's Spark ML pipeline.
- Every removed row changes the test set. Phase 2 kept all 10,000 drivers, so report any rule that removes rows and its effect.

**Shared helpers.** The first code cell runs `00_setup`, which defines the table names (`BRONZE_TABLE`, `SILVER_TABLE`, ...), the file paths (`CLAIMS_CSV`), and `write_delta` / `validate_tables`. Use those names instead of typing table names, so everyone writes to the same place.

In [0]:
%run ./00_setup

In [0]:
from pyspark.sql import functions as F

bronze = spark.table(BRONZE_TABLE)
bronze_rows = bronze.count()

## 1. Inspect and clean

- Parses all rows from Bronze Layer
- Standardizes data types for all features
- Drops all invalid, corrupted, and duplicate records
    - For simplicity, we simply drop all problematic records. In our cases, we do not encounter any. 

### Parse rows

In [0]:
parsed_rows = bronze.filter(F.col("_corrupt_record").isNull())
parsed_counts = parsed_rows.count()
clean = parsed_rows.select(*SOURCE_COLUMNS)

### Ensure data types are correct

In [0]:
for name in SOURCE_COLUMNS:
    value = F.trim(F.col(name))
    clean = clean.withColumn(name, F.when(value != "", value))

categorical_columns = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE",
    "EDUCATION", "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE"
]

for name in categorical_columns:
    clean = clean.withColumn(name, F.lower(F.col(name)))

#### Relabel DRIVING_EXPERIENCE

In [0]:
# shoud be done here because we are regrouping two groups
experience = F.col("DRIVING_EXPERIENCE")
clean = clean.withColumn(
    "DRIVING_EXPERIENCE",
    F.when(experience == "0-9y", "Beginner")
    .when(experience == "10-19y", "Advanced")
    .when(experience.isin("20-29y", "30y+"), "Expert")
    .otherwise(experience)
)

#### Conver to int and doubles

In [0]:
for name in ["MARRIED", "CHILDREN", "VEHICLE_OWNERSHIP", LABEL_COLUMN]:
    value = F.expr(f"try_cast(`{name}` AS DOUBLE)")
    clean = clean.withColumn(name, F.when(value.isin(0.0, 1.0), value.cast("int")))

for name in ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]:
    value = F.expr(f"try_cast(`{name}` AS DOUBLE)")
    valid_count = value.between(0, 2147483647) & ((value%1) == 0)
    clean = clean.withColumn(name, F.when(valid_count, value.cast("int")))

for name in ["CREDIT_SCORE", "ANNUAL_MILEAGE"]:
    value = F.expr(f"try_cast(`{name}` AS DOUBLE)")
    clean = clean.withColumn(name, F.when(F.abs(value) < float("inf"), value))

### Validate values and remove duplicates

In [0]:
valid = clean.filter(
    F.col(ID_COLUMN).isNotNull() & F.col(LABEL_COLUMN).isNotNull()
)
valid_count = valid.count()

In [0]:
silver = valid.dropDuplicates()
silver_count = silver.count()

In [0]:
id_conflicts = silver.groupBy(ID_COLUMN).count().filter(F.col("count") > 1)

if id_conflicts.limit(1).count() > 0:
    display(id_conflicts.limit(20))
    raise ValueError("ID conflicts must be addressed")

## 2. Write and verify the Delta table

In [0]:
print(f"Bronze rows: {bronze.count()}")
print(f"Flagged CSV rows removed: {bronze.count() - parsed_counts}")
print(f"Rows removed for missing ID or invalid outcome: {parsed_counts - valid_count}")
print(f"Duplicate standardized rows removed: {valid_count - silver_count}")
print(f"Silver rows: {silver_count}")

In [0]:
expected_silver_rows = silver.count()

write_delta(silver, SILVER_TABLE)

# Reads the table back from the catalog: format, row count, columns, and ID uniqueness.
validate_tables({SILVER_TABLE: expected_silver_rows})